In [ ]:
%matplotlib inline

import pandas as pd
from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "NNS_2-5"
well_name = '2-5-4'
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
    well_data = project.get_well_data(well_name)

# Lithology Estimation

In [ ]:
import matplotlib.pyplot as plt

from quick_pp.lithology.sand_shale import SandShale
from quick_pp.porosity import *
from quick_pp.qaqc import *
from quick_pp.plotter.plotter import plotly_log, neutron_density_xplot
from quick_pp.rock_type import estimate_vsh_gr
from quick_pp.utils import *

In [ ]:
# Clean up data
well_data = badhole_flagging(well_data)

for col in ['GR', 'RT', 'NPHI', 'RHOB']:
    well_data.loc[:, col] = remove_straights(well_data[col])

# Initialize lithology model
args = {
    'litho_model': 'ss',
    # 'dry_clay_point': (.3, 2.7),
    'wet_clay_point': (0.45, 2.44),
    'hc_corr_angle': neu_den_xplot_hc_correction_angle(rho_water=1.0, rho_hc=0.8, HI_hc=0.9),
    'hc_buffer': 0.01,
}

ss_model = SandShale(**args)
vsand, vcld, _ = ss_model.estimate_lithology(
    nphi=well_data['NPHI'], rhob=well_data['RHOB']
)
args.update(ss_model.__dict__)
# well.update_config(args)  # Save lithology model to well

# Choose to skip HC correction or not
skip_hc_correction = False
if skip_hc_correction is True:
    nphihc, rhobhc = well_data['NPHI'], well_data['RHOB']
else:
    # Implement hydrocarbon correction
    vsh_gr = estimate_vsh_gr(well_data['GR'])
    nphihc, rhobhc, hc_flag = neu_den_xplot_hc_correction(
        well_data['NPHI'], well_data['RHOB'],
        dry_min1_point=args['dry_sand_point'],
        dry_clay_point=args['dry_clay_point'],
        corr_angle=args['hc_corr_angle'], buffer=args['hc_buffer']
    )
    
    # Correct density log
    rhob_corr = den_correction(nphihc, well_data['GR'], vsh_gr=vsh_gr, alpha=0.1)
    badhole_flag =  np.where(abs(well_data['RHOB'] - rhob_corr) > 0.2, 1, 0)
    rhob_corr = np.where((badhole_flag == 1) & (hc_flag == 0), rhob_corr, rhobhc)

    # Estimate lithology
    ss_model = SandShale(**args)
    vsand, vcld, _ = ss_model.estimate_lithology(
        nphi=nphihc, rhob=rhob_corr,
    )

In [ ]:
neutron_density_xplot(well_data['NPHI'], well_data['RHOB'], dry_min1_point=args['dry_sand_point'], **args)

In [ ]:
neutron_density_xplot(nphihc, rhobhc, dry_min1_point=args['dry_sand_point'], **args)

# Porosity Estimation

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import numpy as np

# Estimate porosity
phit = neu_den_xplot_poro(
    nphihc, rhobhc, model='ss',
    dry_min1_point=args['dry_sand_point'],
    dry_clay_point=args['dry_clay_point'],
)

# PHID needs unnormalized lithology
rho_ma = rho_matrix(vsand=vsand, vclay=vcld)
phid = density_porosity(rhobhc, rho_ma)

# Fill missing values in phit with phid
phit = np.where(np.isnan(phit), phid, phit)

# Normalize lithology
volumes = dict(vcld=vcld, vsand=vsand)
volumes = normalize_volumetric(phit, **volumes)
vcld, vsand = volumes['vcld'], volumes['vsand']

# Calculate vclb: volume of clay bound water and phie
phit_shale = estimate_shale_porosity(well_data.NPHI, phid)
vclb = vcld * phit_shale
vclay = vcld + vclb

phie = phit - vclb

In [ ]:
from quick_pp.rock_type import estimate_vsh_gr, estimate_vsh_dn

vsh_gr = estimate_vsh_gr(well_data['GR'])
vsh_nd = estimate_vsh_dn(well_data.NPHI, phid)
fig, axs = plt.subplots(3, 1, figsize=(20, 5), sharex=True)
axs[0].plot(well_data.DEPTH, phit, label='PHIT')
axs[0].plot(well_data.DEPTH, phid, label='PHID')
axs[0].scatter(well_data.DEPTH, well_data.CPORE, label='CPORE' , marker='.', color='black')
axs[0].set_ylim(0, .5)
axs[0].legend()

axs[1].plot(well_data.DEPTH, vsh_gr, label='vsh_gr')
axs[1].plot(well_data.DEPTH, vsh_nd, label='vsh_nd')
axs[1].plot(well_data.DEPTH, vclay, label='vclay')
axs[1].set_ylim(-.1, 1.1)
axs[1].legend()

axs[2].plot(well_data.DEPTH, well_data['GR'], label='GR')
axs[2].set_ylim(0,250)
axs[2].legend()

# Plotting the result

In [ ]:
well_data['NPHI_HC'] = nphihc
well_data['RHOB_HC'] = rhobhc
well_data['VCLAY'] = vclay
well_data['VSAND'] = vsand
well_data['PHIT'] = phit
well_data['PHID'] = phid

# Plot the results
fig = plotly_log(well_data, well_name=well_name, depth_uom='m')
fig.show(config=dict(scrollZoom=True))
# fig.write_html(rf"{well_name}_log.html", config=dict(scrollZoom=True))

# Apply to all

In [ ]:
from tqdm import tqdm

# Initialize lithology model
final_args = {
    'litho_model': 'ss',
    # 'dry_clay_point': (.3, 2.7),
    'wet_clay_point': (0.45, 2.45),
    'hc_corr_angle': neu_den_xplot_hc_correction_angle(rho_water=1.0, rho_hc=0.8, HI_hc=0.9),
    'hc_buffer': 0.01,
}

for well_name, well_data in tqdm(df.groupby('WELL_NAME'), desc='Estimating for all wells'):
    tqdm.write(f'Processing {well_name}: {len(well_data)} rows')
    # Clean up data
    well_data = badhole_flagging(well_data)

    for col in ['GR', 'RT', 'NPHI', 'RHOB']:
        well_data.loc[:, col] = remove_straights(well_data[col])

    ss_model = SandShale(**final_args)
    vsand, vcld, _ = ss_model.estimate_lithology(
        nphi=well_data['NPHI'], rhob=well_data['RHOB']
    )
    final_args.update(ss_model.__dict__)

    # Implement hydrocarbon correction
    vsh_gr = estimate_vsh_gr(well_data['GR'])
    nphihc, rhobhc, hc_flag = neu_den_xplot_hc_correction(
        well_data['NPHI'], well_data['RHOB'], vsh_gr=vsh_gr,
        dry_min1_point=final_args['dry_sand_point'],
        dry_clay_point=final_args['dry_clay_point'],
        corr_angle=final_args['hc_corr_angle'], buffer=final_args['hc_buffer']
    )
    
    # Correct density log
    rhob_corr = den_correction(nphihc, well_data['GR'], vsh_gr=vsh_gr, alpha=0.1)
    badhole_flag =  np.where(abs(well_data['RHOB'] - rhob_corr) > 0.2, 1, 0)
    rhob_corr = np.where((badhole_flag == 1) & (hc_flag == 0), rhob_corr, rhobhc)

    # Estimate lithology
    ss_model = SandShale(**final_args)
    vsand, vcld, _ = ss_model.estimate_lithology(
        nphi=nphihc, rhob=rhob_corr,
    )

    # Estimate porosity
    phit = neu_den_xplot_poro(
        nphihc, rhobhc, model='ss',
        dry_min1_point=final_args['dry_sand_point'],
        dry_clay_point=final_args['dry_clay_point'],
    )
    
    rho_ma = rho_matrix(vsand=vsand, vclay=vcld)
    phid = density_porosity(rhobhc, rho_ma)

    # Fill missing values in phit with phid
    phit = np.where(np.isnan(phit), phid, phit)

    # Normalize lithology
    volumes = dict(vcld=vcld, vsand=vsand)
    volumes = normalize_volumetric(phit, **volumes)
    vclay, vsand = volumes['vcld'], volumes['vsand']

    # Calculate vclb: volume of clay bound water and phie
    phit_shale = estimate_shale_porosity(well_data.NPHI, phid)
    vclb = vclay * phit_shale
    phie = phit - vclb

    well_data['NPHI_HC'] = nphihc
    well_data['RHOB_HC'] = rhobhc
    well_data['VCLAY'] = vclay
    well_data['VSAND'] = vsand
    well_data['PHIT'] = phit
    well_data['PHIE'] = phie.clip(0, 1)

    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(well_data, well_configs={well_name: final_args})
        project.save()

## PHIT vs. CPORE Validation

Initial comparison indicates a poor match with MAPE of around 30%. The core points in

In [ ]:
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    score_df = project.get_all_data()
score_df = score_df[['WELL_NAME', 'CPORE', 'PHIT']].copy()
score_df.dropna(inplace=True)
mape = round(mean_absolute_percentage_error(score_df.CPORE, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE, score_df.PHIT), 2)
print(f"\n ### PHIT MAPE: {mape:.2f}")
print(f" ### PHIT R2: {r2:.2f}")

plt.scatter(score_df.CPORE, score_df.PHIT, label=f'Overall - R2: {r2}, MAPE: {mape}')
for well, data in score_df.groupby('WELL_NAME'):
    mape = round(mean_absolute_percentage_error(data.CPORE, data.PHIT), 2)
    r2 = round(r2_score(data.CPORE, data.PHIT), 2)
    plt.scatter(data.CPORE, data.PHIT, label=f'{well} - R2: {r2}, MAPE: {mape}')
plt.xlabel('Actual')
plt.ylabel('Calculated')
plt.xlim(0, .5)
plt.ylim(0, .5)
plt.legend()

In [ ]:
STOP

In [ ]:
# Compare the score between shifted and non shifted.
score_df = merged_df[['WELL_NAME', 'CPORE', 'CPORE_SHIFTED', 'PHIT']].copy()
score_df.dropna(inplace=True)

mape = round(mean_absolute_percentage_error(score_df.CPORE, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE, score_df.PHIT), 2)
plt.scatter(score_df.CPORE, score_df.PHIT, label=f'Overall Original Data - R2: {r2}, MAPE: {mape}')

mape = round(mean_absolute_percentage_error(score_df.CPORE_SHIFTED, score_df.PHIT), 2)
r2 = round(r2_score(score_df.CPORE_SHIFTED, score_df.PHIT), 2)
plt.scatter(score_df.CPORE_SHIFTED, score_df.PHIT, label=f'Shifted Data - R2: {r2}, MAPE: {mape}')

for well, data in score_df.groupby('WELL_NAME'):
    mape = round(mean_absolute_percentage_error(data.CPORE_SHIFTED, data.PHIT), 2)
    r2 = round(r2_score(data.CPORE_SHIFTED, data.PHIT), 2)
    plt.scatter(data.CPORE_SHIFTED, data.PHIT, label=f'{well} - R2: {r2}, MAPE: {mape}')
plt.xlabel('Actual')
plt.ylabel('Calculated')
plt.xlim(0, .5)
plt.ylim(0, .5)
plt.legend()

In [ ]:
# Need to perform core depth correction

In [ ]:
copy_df = merged_df[merged_df.WELL_NAME == '2-5-4']
plt.figure(figsize=(25, 2))
plt.scatter(copy_df.DEPTH, copy_df.CPORE, c='green', marker='.', label='CPORE ORI')
plt.scatter(copy_df.DEPTH, copy_df.CPORE_SHIFTED, c='r', marker='x', label='CPORE SHIFTED')
plt.plot(copy_df.DEPTH, copy_df.PHIT, 'b--', label='PHIT')

# --- Annotation Loop ---
# Iterate over each row in the DataFrame to add annotations
cores = copy_df.dropna(subset='CORE_ID')
for index, row in cores.iterrows():
    plt.annotate(
        text=row['CORE_ID'],                      # The text to display (the CORE_ID)
        xy=(row['DEPTH'], row['CPORE']),  # The point to annotate (x, y)
        xytext=(5, 5),                            # The position of the text (x, y offset)
        textcoords='offset points',               # Interpret xytext as an offset from the point
        ha='left',                                # Horizontal alignment
        va='bottom',                              # Vertical alignment
        fontsize=8,
        arrowprops=dict(arrowstyle='->', color='gray') # Optional: adds an arrow
    )
# --- End of Annotation Loop ---

plt.legend()
plt.ylim(0, .4)
plt.xlim(3060, 3110)

In [ ]:
# Save result to database
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    project.update_data(well_data, well_configs={well_name: final_args})
    project.save()